In [1]:
# ==============================
# Tanzila Islam
# Email: tanzilamohita@gmail.com
# ===============================

#### Import Libraries

In [1]:
import numpy as np
import pandas as pd
from Data_Processing import one_hot_encode_snp_array
from DeepCGP_ import compress_data

import warnings
warnings.filterwarnings("ignore")

GPU is NOT available. Running on CPU.


#### Example use of Data Compression with Demo Data

In [8]:
# read data
data = pd.read_csv(f'../Data/X.csv', index_col=0)
print(data.iloc[:5, :5])

              V1 V2 V3 V4 V5
PURPLE         A  G  A  A  A
KinShanZim     A  G  A  A  G
N32            A  G  C  A  G
Dular          A  G  C  A  G
HashikalmiAus  A  G  C  A  G


In [9]:
# convert to one hot encoding
snp_array = data.astype(str).values
one_hot = one_hot_encode_snp_array(snp_array)
print(one_hot)
print(one_hot.shape)

[[1 0 0 ... 0 1 0]
 [1 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]]
(189, 56)


In [10]:
best_config = {
    "InputDim": 28,
    "Compress": [14, 7, 3],
    "BatchSize": 52,
    "LearningRate": 0.001,
    "Epochs": 200,
}

X = one_hot.astype(np.float32)
compressed = compress_data(X, best_config, seed=42, verbose=True)

→ Chunk 1/2 | chunk shape=(189, 28)
→ Chunk 2/2 | chunk shape=(189, 28)
Final compressed shape: (189, 6)


#### Example use of ConvCGP Model on fixed training-validation split

In [12]:
C2 = pd.read_csv(f'../Data/X_C2.csv', index_col=0)
print(C2.head())
Y = pd.read_csv(f'../Data/Y.csv', index_col=0)
print(Y.head())

               V1  V2  V3  V4  V5  V6  V7  V8  V9  V10  ...  V416  V417  V418  \
PURPLE          0   1   0   1   1   0   1   1   1    1  ...     0     1     1   
KinShanZim      1   1   0   0   1   1   0   1   1    1  ...     1     0     1   
N32             1   1   1   0   1   0   1   1   1    1  ...     1     0     1   
Dular           0   1   0   0   1   0   1   0   1    1  ...     1     0     1   
HashikalmiAus   1   1   1   0   1   0   1   1   1    1  ...     1     1     1   

               V419  V420  V421  V422  V423  V424  V425  
PURPLE            1     1     1     1     1     1     0  
KinShanZim        1     1     1     1     1     1     0  
N32               1     1     1     1     1     0     0  
Dular             1     1     1     1     1     0     0  
HashikalmiAus     1     1     1     1     1     0     0  

[5 rows x 425 columns]
                   x
PURPLE          75.5
KinShanZim     103.5
N32             95.7
Dular          101.0
HashikalmiAus   77.3


In [22]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from ConvCGP import create_model
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Drop missing values
Y = Y.dropna()
X = C2.loc[Y.index]

# Preprocess X
X.columns = np.arange(0, len(X.columns))
Y.columns = np.arange(0, len(Y.columns))

# Define callbacks
es = EarlyStopping(monitor='val_loss', mode='min', patience=50, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=20, min_lr=1e-5)

corr_df = []

# Training and evaluating on X
for i in range(0, 1):

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, Y[i], test_size=0.2, random_state=42
    )

    X_train = np.expand_dims(X_train, axis=2)
    X_valid = np.expand_dims(X_valid, axis=2)

    model_X = create_model(input_shape=(X_train.shape[1], 1))

    model_X.fit(
        X_train, y_train,
        epochs=400,
        batch_size=64,
        validation_data=(X_valid, y_valid),
        callbacks=[es, reduce_lr],
        shuffle=False,
        verbose=0
    )

    # Prediction
    y_hat_X = model_X.predict(X_valid, verbose=0)

    corr_X = np.corrcoef(y_valid, y_hat_X[:, 0])[0, 1]

    print(f'Correlation for trait {i} = {corr_X:.4f}')

    corr_df.append([i, corr_X])

Correlation for trait 0 = 0.7187
